In [5]:
import numpy as np
import pandas as pd
import os
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [6]:
name_path = 'hcm_aqi_full_dataset.csv'
folder_path = 'data'

file_path = os.path.join(folder_path, name_path)

df = pd.read_csv(file_path) 
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])
df.head(5)


,time,PM10,PM2.5,CO,NO2,O3,SO2,AQI,UV,Temperature,Humidity,Rain,Wind_Speed,Wind_Dir
0,2023-01-01 00:00:00,95.0,65.0,993.0,84.4,25.0,37.5,133,0.0,23.5,65,0.0,11.3,9
1,2023-01-01 01:00:00,86.1,59.0,846.0,70.2,28.0,30.8,133,0.0,23.0,67,0.0,10.3,12
2,2023-01-01 02:00:00,83.4,57.0,821.0,65.8,26.0,28.9,132,0.0,22.5,70,0.0,7.9,360
3,2023-01-01 03:00:00,80.1,54.8,834.0,64.0,22.0,28.5,131,0.0,22.0,73,0.0,8.3,360
4,2023-01-01 04:00:00,69.4,47.5,838.0,60.1,20.0,27.2,129,0.0,21.9,72,0.0,7.9,357


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler


df['time'] = pd.to_datetime(df['time'])
df['hour'] = df['time'].dt.hour
df['dayofyear'] = df['time'].dt.dayofyear

df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['doy_sin'] = np.sin(2 * np.pi * df['dayofyear'] / 365.25)
df['doy_cos'] = np.cos(2 * np.pi * df['dayofyear'] / 365.25)
df['wind_sin'] = np.sin(2 * np.pi * df['Wind_Dir'] / 360)
df['wind_cos'] = np.cos(2 * np.pi * df['Wind_Dir'] / 360)


features = ['PM2.5', 'PM10', 'NO2', 'CO', 'SO2', 'Temperature', 'Humidity', 'Rain', 'Wind_Speed', 
            'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos', 'wind_sin', 'wind_cos']


feature_scaler = MinMaxScaler()
scaled_data = feature_scaler.fit_transform(df[features])

target_scaler = MinMaxScaler()
target_idx = features.index('PM2.5')
scaled_target = target_scaler.fit_transform(df[['PM2.5']])

In [ ]:
def create_lstm_dataset(data, target, window_size=48, forecast_steps=3):
    X, y = [], []
    for i in range(len(data) - window_size - forecast_steps + 1):
        # Input: 48 giờ quá khứ của TẤT CẢ features
        X.append(data[i : i + window_size, :])
        # Output: 3 giờ tương lai của CỘT ĐÍCH (PM2.5)
        y.append(target[i + window_size : i + window_size + forecast_steps].flatten())
        
    return np.array(X), np.array(y)

# Lấy 48 tiếng quá khứ để dự đoán 3 tiếng tiếp theo
window_size = 48
forecast_steps = 3

X_all, y_all = create_lstm_dataset(scaled_data, scaled_target, window_size, forecast_steps)

print(f"Shape của X: {X_all.shape}") 
print(f"Shape của y: {y_all.shape}") 

Shape của X: (26830, 48, 15)
Shape của y: (26830, 3)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

train_size = int(len(X_all) * 0.8)

X_train, X_test = X_all[:train_size], X_all[train_size:]
y_train, y_test = y_all[:train_size], y_all[train_size:]

# 2. KIẾN TRÚC LSTM 
model = Sequential([
    LSTM(units=64, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2), 
    LSTM(units=32, return_sequences=False),
    Dropout(0.2),
    

    Dense(units=forecast_steps)
])

model.compile(optimizer='adam', loss='mse')
model.summary()



Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 48, 64)            20480     
                                                                 
 dropout (Dropout)           (None, 48, 64)            0         
                                                                 
 lstm_1 (LSTM)               (None, 32)                12416     
                                                                 
 dropout_1 (Dropout)         (None, 32)                0         
                                                                 
 dense (Dense)               (None, 3)                 99        
                                                                 
Total params: 32995 (128.89 KB)
Trainable params: 32995 (128.89 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=5, 
    restore_best_weights=True,
    verbose=1
)
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.5, 
    patience=3, 
    min_lr=1e-5,
    verbose=1
)
print("🚀 Bắt đầu huấn luyện LSTM với Early Stopping...")

history = model.fit(
    X_train, y_train,
    epochs=100, 
    batch_size=64, # Xử lý 64 mẫu mỗi lần cập nhật trọng số
    validation_data=(X_test, y_test),
    callbacks=[early_stop, reduce_lr], #
    verbose=1
)

print("✅ Đã train xong! Model hội tụ an toàn mà không bị Overfitting.")

🚀 Bắt đầu huấn luyện LSTM với Early Stopping...
Epoch 1/100

336/336 [==============================] - 26s 63ms/step - loss: 0.0076 - val_loss: 0.0076 - lr: 0.0010
Epoch 2/100
336/336 [==============================] - 21s 61ms/step - loss: 0.0044 - val_loss: 0.0054 - lr: 0.0010
Epoch 3/100
336/336 [==============================] - 24s 73ms/step - loss: 0.0034 - val_loss: 0.0045 - lr: 0.0010
Epoch 4/100
336/336 [==============================] - 37s 111ms/step - loss: 0.0030 - val_loss: 0.0043 - lr: 0.0010
Epoch 5/100
336/336 [==============================] - 32s 95ms/step - loss: 0.0027 - val_loss: 0.0037 - lr: 0.0010
Epoch 6/100
336/336 [==============================] - 39s 117ms/step - loss: 0.0025 - val_loss: 0.0034 - lr: 0.0010
Epoch 7/100
336/336 [==============================] - 37s 110ms/step - loss: 0.0024 - val_loss: 0.0036 - lr: 0.0010
Epoch 8/100
336/336 [==============================] - 24s 73ms/step - loss: 0.0023 - val_loss: 0.0031 - lr: 0.0010
Epoch 9/100
336/336 

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

print("🚀 Đang tính toán kết quả trên tập Test...")
# Dự báo trên tập Test
y_pred_scaled = model.predict(X_test)

# Inverse Transform để trả về số thực (microgram/m3)
y_pred_real = target_scaler.inverse_transform(y_pred_scaled)
y_test_real = target_scaler.inverse_transform(y_test)

# Vòng lặp tính toán cho cả t+1, t+2, t+3
for i in range(3):
    t_pred = y_pred_real[:, i]
    t_true = y_test_real[:, i]
    
    r2 = r2_score(t_true, t_pred)
    mae = mean_absolute_error(t_true, t_pred)
    rmse = np.sqrt(mean_squared_error(t_true, t_pred))
    
    print(f"--- KẾT QUẢ CHO t+{i+1} ---")
    print(f"R2 Score: {r2:.4f}")
    print(f"MAE:      {mae:.4f}")
    print(f"RMSE:     {rmse:.4f}\n")

🚀 Đang tính toán kết quả trên tập Test...
168/168 [==============================] - 2s 10ms/step
--- KẾT QUẢ CHO t+1 ---
R2 Score: 0.9266
MAE:      2.8117
RMSE:     4.2060

--- KẾT QUẢ CHO t+2 ---
R2 Score: 0.8400
MAE:      4.3119
RMSE:     6.2040

--- KẾT QUẢ CHO t+3 ---
R2 Score: 0.7442
MAE:      5.5661
RMSE:     7.8443

